In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
FILE_PATH = '/content/drive/My Drive/train.csv'

In [ ]:
CHUNK_SIZE = 200_000
SAMPLE_RATE = 0.10
chunks = []
for chunk in pd.read_csv(FILE_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    sampled = chunk.sample(frac=SAMPLE_RATE, random_state=42)
    chunks.append(sampled)

df_raw = pd.concat(chunks, ignore_index=True)

In [ ]:
print(df_raw.columns)

Index(['fecha_dato', 'ncodpers', 'ind_empleado', 'pais_residencia', 'sexo',
       'age', 'fecha_alta', 'ind_nuevo', 'antiguedad', 'indrel',
       'ult_fec_cli_1t', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'indext',
       'conyuemp', 'canal_entrada', 'indfall', 'tipodom', 'cod_prov',
       'nomprov', 'ind_actividad_cliente', 'renta', 'segmento',
       'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1',
       'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1',
       'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1',
       'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1',
       'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1',
       'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1',
       'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1',
       'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1'],
      dtype='object')


In [ ]:
df_raw.rename(columns={

    # Customer information
    'fecha_dato': 'date',
    'ncodpers': 'customer_id',
    'ind_empleado': 'employee_index',
    'pais_residencia': 'country',
    'sexo': 'gender',
    'age': 'age',
    'fecha_alta': 'join_date',
    'ind_nuevo': 'new_customer',
    'antiguedad': 'seniority',
    'indrel': 'customer_type',
    'ult_fec_cli_1t': 'last_primary_date',
    'indrel_1mes': 'customer_relation_month_start',
    'tiprel_1mes': 'relation_type',
    'indresi': 'resident',
    'indext': 'foreigner',
    'conyuemp': 'employee_spouse',
    'canal_entrada': 'join_channel',
    'indfall': 'deceased',
    'tipodom': 'address_type',
    'cod_prov': 'province_code',
    'nomprov': 'province',
    'ind_actividad_cliente': 'active_customer',
    'renta': 'income',
    'segmento': 'segment',

    # Product columns
    'ind_ahor_fin_ult1': 'product_savings_account',
    'ind_aval_fin_ult1': 'product_guarantees',
    'ind_cco_fin_ult1': 'product_current_account',
    'ind_cder_fin_ult1': 'product_derivada_account',
    'ind_cno_fin_ult1': 'product_payroll_account',
    'ind_ctju_fin_ult1': 'product_junior_account',
    'ind_ctma_fin_ult1': 'product_mas_particular_account',
    'ind_ctop_fin_ult1': 'product_particular_account',
    'ind_ctpp_fin_ult1': 'product_particular_plus_account',
    'ind_deco_fin_ult1': 'product_short_term_deposit',
    'ind_deme_fin_ult1': 'product_medium_term_deposit',
    'ind_dela_fin_ult1': 'product_long_term_deposit',
    'ind_ecue_fin_ult1': 'product_e_account',
    'ind_fond_fin_ult1': 'product_funds',
    'ind_hip_fin_ult1': 'product_mortgage',
    'ind_plan_fin_ult1': 'product_pension_plan',
    'ind_pres_fin_ult1': 'product_loans',
    'ind_reca_fin_ult1': 'product_taxes',
    'ind_tjcr_fin_ult1': 'product_credit_card',
    'ind_valo_fin_ult1': 'product_securities',
    'ind_viv_fin_ult1': 'product_home_account',
    'ind_nomina_ult1': 'product_payroll',
    'ind_nom_pens_ult1': 'product_pension',
    'ind_recibo_ult1': 'product_direct_debit'
}, inplace=True)

In [ ]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1271586 entries, 0 to 1271585
Data columns (total 48 columns):
 #   Column                           Non-Null Count    Dtype  
---  ------                           --------------    -----  
 0   date                             1271586 non-null  object 
 1   customer_id                      1271586 non-null  int64  
 2   employee_index                   1268792 non-null  object 
 3   country                          1268792 non-null  object 
 4   gender                           1268788 non-null  object 
 5   age                              1271586 non-null  object 
 6   join_date                        1268792 non-null  object 
 7   new_customer                     1268792 non-null  float64
 8   seniority                        1271586 non-null  object 
 9   customer_type                    1268792 non-null  float64
 10  last_primary_date                2275 non-null     object 
 11  customer_relation_month_start    1257136 non-null 

In [ ]:
df_raw.drop(['last_primary_date', 'address_type', 'employee_spouse', "customer_type", "resident", "employee_index", "deceased", "country",], axis=1, errors='ignore', inplace=True)

In [ ]:
df = df_raw.replace([' NA', 'NA', ' ', ''], np.nan)
df=df.dropna()
# renta: median theo tỉnh
df['income'] = pd.to_numeric(df['income'], errors='coerce')
df['income'] = df['income'].fillna(df.groupby('province')['income'].transform('median'))
# age: (18-100) + median
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df.loc[df['age'] < 18, 'age'] = df['age'].median()
df.loc[df['age'] > 100, 'age'] = df['age'].median()
df['age'] = df['age'].fillna(df['age'].median())
# Thâm niên
df['seniority'] = pd.to_numeric(df['seniority'], errors='coerce')
df.loc[df['seniority'] < 0, 'seniority'] = 0
df['seniority'] = df['seniority'].fillna(0)
# Category
category_cols = ['gender',
               'customer_relation_month_start', 'relation_type', 'foreigner', 'join_date',
               'join_channel', 'province', 'segment', 'new_customer', 'province_code',
                 'active_customer' ]
for col in category_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)
# Product
product_cols = [c for c in df.columns if c.startswith('product_')]
df[product_cols] = (df[product_cols]
                    .apply(pd.to_numeric, errors='coerce')
                    .fillna(0)
                    .astype('int8'))

/tmp/ipykernel_4219/4025126162.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [ ]:
#encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
category_cols=['gender', 'segment', 'province']
for col in category_cols:
  df[col]=le.fit_transform(df[col].astype(str))

In [ ]:
df.isnull().sum()

,0
date,0
customer_id,0
gender,0
age,0
join_date,0
new_customer,0
seniority,0
customer_relation_month_start,0
relation_type,0
foreigner,0


In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['join_date'] = pd.to_datetime(df['join_date'])
df['age'] = df['age'].astype(np.int8)
df['seniority'] = df['seniority'].astype(np.int16)
df['gender'] = df['gender'].astype(np.int8)
df['province'] = df['province'].astype(np.int16)
df['segment'] = df['segment'].astype(np.int8)
df['customer_id'] = df['customer_id'].astype(np.int32)
df['new_customer'] = df['new_customer'].astype(np.int8)
df['active_customer'] = df['active_customer'].astype(np.int8)
df['province_code'] = df['province_code'].astype(np.int16)
other_category_cols = ['customer_relation_month_start', 'relation_type',
                       'foreigner', 'join_channel']
for col in other_category_cols:
  df[col]=le.fit_transform(df[col].astype(str))
  df[col]=df[col].astype(np.int16)
df.sort_values(by=['customer_id', 'date'], inplace=True)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1010228 entries, 1109860 to 1251206
Data columns (total 40 columns):
 #   Column                           Non-Null Count    Dtype         
---  ------                           --------------    -----         
 0   date                             1010228 non-null  datetime64[ns]
 1   customer_id                      1010228 non-null  int32         
 2   gender                           1010228 non-null  int8          
 3   age                              1010228 non-null  int8          
 4   join_date                        1010228 non-null  datetime64[ns]
 5   new_customer                     1010228 non-null  int8          
 6   seniority                        1010228 non-null  int16         
 7   customer_relation_month_start    1010228 non-null  int16         
 8   relation_type                    1010228 non-null  int16         
 9   foreigner                        1010228 non-null  int16         
 10  join_channel                 

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1010228 entries, 1109860 to 1251206
Data columns (total 40 columns):
 #   Column                           Non-Null Count    Dtype         
---  ------                           --------------    -----         
 0   date                             1010228 non-null  datetime64[ns]
 1   customer_id                      1010228 non-null  int32         
 2   gender                           1010228 non-null  int8          
 3   age                              1010228 non-null  int8          
 4   join_date                        1010228 non-null  datetime64[ns]
 5   new_customer                     1010228 non-null  int8          
 6   seniority                        1010228 non-null  int16         
 7   customer_relation_month_start    1010228 non-null  int16         
 8   relation_type                    1010228 non-null  int16         
 9   foreigner                        1010228 non-null  int16         
 10  join_channel                 

In [ ]:
df.head()

,date,customer_id,gender,age,join_date,new_customer,seniority,customer_relation_month_start,relation_type,foreigner,...,product_mortgage,product_pension_plan,product_loans,product_taxes,product_credit_card,product_securities,product_home_account,product_payroll,product_pension,product_direct_debit
1109860,2016-03-28,15890,1,63,1995-01-16,0,254,1,0,0,...,0,1,0,0,1,0,0,1,1,1
282028,2015-05-28,15892,0,61,1995-01-16,0,246,1,0,0,...,0,0,0,1,1,1,0,0,0,1
419704,2015-07-28,15892,0,61,1995-01-16,0,246,0,0,0,...,0,0,0,1,1,1,0,0,0,1
869442,2015-12-28,15893,1,62,1997-10-03,0,251,0,0,0,...,0,0,0,0,0,1,0,0,0,0
281848,2015-05-28,15894,1,59,1995-01-16,0,246,1,0,0,...,0,0,0,1,1,1,0,1,1,1


In [ ]:
TEST_PATH = '/content/drive/My Drive/test.csv'

In [ ]:
CHUNK_SIZE = 200_000
SAMPLE_RATE = 0.10
chunks = []
for chunk in pd.read_csv(FILE_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    sampled = chunk.sample(frac=SAMPLE_RATE, random_state=42)
    chunks.append(sampled)

df_test = pd.concat(chunks, ignore_index=True)

In [ ]:
print(df_test.columns)

Index(['fecha_dato', 'ncodpers', 'ind_empleado', 'pais_residencia', 'sexo',
       'age', 'fecha_alta', 'ind_nuevo', 'antiguedad', 'indrel',
       'ult_fec_cli_1t', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'indext',
       'conyuemp', 'canal_entrada', 'indfall', 'tipodom', 'cod_prov',
       'nomprov', 'ind_actividad_cliente', 'renta', 'segmento',
       'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1',
       'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1',
       'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1',
       'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1',
       'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1',
       'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1',
       'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1',
       'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1'],
      dtype='object')


In [ ]:
df_test.rename(columns={

    # Customer information
    'fecha_dato': 'date',
    'ncodpers': 'customer_id',
    'ind_empleado': 'employee_index',
    'pais_residencia': 'country',
    'sexo': 'gender',
    'age': 'age',
    'fecha_alta': 'join_date',
    'ind_nuevo': 'new_customer',
    'antiguedad': 'seniority',
    'indrel': 'customer_type',
    'ult_fec_cli_1t': 'last_primary_date',
    'indrel_1mes': 'customer_relation_month_start',
    'tiprel_1mes': 'relation_type',
    'indresi': 'resident',
    'indext': 'foreigner',
    'conyuemp': 'employee_spouse',
    'canal_entrada': 'join_channel',
    'indfall': 'deceased',
    'tipodom': 'address_type',
    'cod_prov': 'province_code',
    'nomprov': 'province',
    'ind_actividad_cliente': 'active_customer',
    'renta': 'income',
    'segmento': 'segment',

    # Product columns
    'ind_ahor_fin_ult1': 'product_savings_account',
    'ind_aval_fin_ult1': 'product_guarantees',
    'ind_cco_fin_ult1': 'product_current_account',
    'ind_cder_fin_ult1': 'product_derivada_account',
    'ind_cno_fin_ult1': 'product_payroll_account',
    'ind_ctju_fin_ult1': 'product_junior_account',
    'ind_ctma_fin_ult1': 'product_mas_particular_account',
    'ind_ctop_fin_ult1': 'product_particular_account',
    'ind_ctpp_fin_ult1': 'product_particular_plus_account',
    'ind_deco_fin_ult1': 'product_short_term_deposit',
    'ind_deme_fin_ult1': 'product_medium_term_deposit',
    'ind_dela_fin_ult1': 'product_long_term_deposit',
    'ind_ecue_fin_ult1': 'product_e_account',
    'ind_fond_fin_ult1': 'product_funds',
    'ind_hip_fin_ult1': 'product_mortgage',
    'ind_plan_fin_ult1': 'product_pension_plan',
    'ind_pres_fin_ult1': 'product_loans',
    'ind_reca_fin_ult1': 'product_taxes',
    'ind_tjcr_fin_ult1': 'product_credit_card',
    'ind_valo_fin_ult1': 'product_securities',
    'ind_viv_fin_ult1': 'product_home_account',
    'ind_nomina_ult1': 'product_payroll',
    'ind_nom_pens_ult1': 'product_pension',
    'ind_recibo_ult1': 'product_direct_debit'
}, inplace=True)

In [ ]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1271586 entries, 0 to 1271585
Data columns (total 48 columns):
 #   Column                           Non-Null Count    Dtype  
---  ------                           --------------    -----  
 0   date                             1271586 non-null  object 
 1   customer_id                      1271586 non-null  int64  
 2   employee_index                   1268792 non-null  object 
 3   country                          1268792 non-null  object 
 4   gender                           1268788 non-null  object 
 5   age                              1271586 non-null  object 
 6   join_date                        1268792 non-null  object 
 7   new_customer                     1268792 non-null  float64
 8   seniority                        1271586 non-null  object 
 9   customer_type                    1268792 non-null  float64
 10  last_primary_date                2275 non-null     object 
 11  customer_relation_month_start    1257136 non-null 

In [ ]:
df_test.drop(['last_primary_date', 'address_type', 'employee_spouse', "customer_type", "resident", "employee_index", "deceased", "country",], axis=1, errors='ignore', inplace=True)

In [ ]:
df_test = df_test.replace([' NA', 'NA', ' ', ''], np.nan)
df_test=df_test.dropna()
# renta: median theo tỉnh
df_test['income'] = pd.to_numeric(df_test['income'], errors='coerce')
df_test['income'] = df_test['income'].fillna(df_test.groupby('province')['income'].transform('median'))
# age: (18-100) + median
df_test['age'] = pd.to_numeric(df_test['age'], errors='coerce')
df_test.loc[df['age'] < 18, 'age'] = df_test['age'].median()
df_test.loc[df['age'] > 100, 'age'] = df_test['age'].median()
df_test['age'] = df_test['age'].fillna(df_test['age'].median())
# Thâm niên
df_test['seniority'] = pd.to_numeric(df_test['seniority'], errors='coerce')
df_test.loc[df_test['seniority'] < 0, 'seniority'] = 0
df_test['seniority'] = df_test['seniority'].fillna(0)
# Category
category_cols = ['gender',
               'customer_relation_month_start', 'relation_type', 'foreigner', 'join_date',
               'join_channel', 'province', 'segment', 'new_customer', 'province_code',
                 'active_customer' ]
for col in category_cols:
    df_test[col].fillna(df_test[col].mode()[0], inplace=True)
# Product
product_cols = [c for c in df_test.columns if c.startswith('product_')]
df_test[product_cols] = (df_test[product_cols]
                    .apply(pd.to_numeric, errors='coerce')
                    .fillna(0)
                    .astype('int8'))

/tmp/ipykernel_4219/3675488897.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_test[col].fillna(df_test[col].mode()[0], inplace=True)


In [ ]:
for col in category_cols:
  df_test[col]=le.fit_transform(df_test[col].astype(str))

In [ ]:
df_test.isnull().sum()

,0
date,0
customer_id,0
gender,0
age,0
join_date,0
new_customer,0
seniority,0
customer_relation_month_start,0
relation_type,0
foreigner,0


In [ ]:
df_test['date'] = pd.to_datetime(df_test['date'])
df_test['join_date'] = pd.to_datetime(df_test['join_date'])
df_test['age'] = df_test['age'].astype(np.int8)
df_test['seniority'] = df_test['seniority'].astype(np.int16)
df_test['gender'] = df_test['gender'].astype(np.int8)
df_test['province'] = df_test['province'].astype(np.int16)
df_test['segment'] = df_test['segment'].astype(np.int8)
df_test['customer_id'] = df_test['customer_id'].astype(np.int32)
df_test['new_customer'] = df_test['new_customer'].astype(np.int8)
df_test['active_customer'] = df_test['active_customer'].astype(np.int8)
df_test['province_code'] = df_test['province_code'].astype(np.int16)
for col in other_category_cols:
  df_test[col]=le.fit_transform(df_test[col].astype(str))
  df_test[col]=df_test[col].astype(np.int16)
df_test.sort_values(by=['customer_id', 'date'], inplace=True)
print(df_test.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1010228 entries, 1109860 to 1251206
Data columns (total 40 columns):
 #   Column                           Non-Null Count    Dtype         
---  ------                           --------------    -----         
 0   date                             1010228 non-null  datetime64[ns]
 1   customer_id                      1010228 non-null  int32         
 2   gender                           1010228 non-null  int8          
 3   age                              1010228 non-null  int8          
 4   join_date                        1010228 non-null  datetime64[ns]
 5   new_customer                     1010228 non-null  int8          
 6   seniority                        1010228 non-null  int16         
 7   customer_relation_month_start    1010228 non-null  int16         
 8   relation_type                    1010228 non-null  int16         
 9   foreigner                        1010228 non-null  int16         
 10  join_channel                 

In [ ]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1010228 entries, 1109860 to 1251206
Data columns (total 40 columns):
 #   Column                           Non-Null Count    Dtype         
---  ------                           --------------    -----         
 0   date                             1010228 non-null  datetime64[ns]
 1   customer_id                      1010228 non-null  int32         
 2   gender                           1010228 non-null  int8          
 3   age                              1010228 non-null  int8          
 4   join_date                        1010228 non-null  datetime64[ns]
 5   new_customer                     1010228 non-null  int8          
 6   seniority                        1010228 non-null  int16         
 7   customer_relation_month_start    1010228 non-null  int16         
 8   relation_type                    1010228 non-null  int16         
 9   foreigner                        1010228 non-null  int16         
 10  join_channel                 

In [ ]:
df_test.head()

,date,customer_id,gender,age,join_date,new_customer,seniority,customer_relation_month_start,relation_type,foreigner,...,product_mortgage,product_pension_plan,product_loans,product_taxes,product_credit_card,product_securities,product_home_account,product_payroll,product_pension,product_direct_debit
1109860,2016-03-28,15890,1,63,1970-01-01 00:00:00.000000000,0,254,1,0,0,...,0,1,0,0,1,0,0,1,1,1
282028,2015-05-28,15892,0,61,1970-01-01 00:00:00.000000000,0,246,1,0,0,...,0,0,0,1,1,1,0,0,0,1
419704,2015-07-28,15892,0,61,1970-01-01 00:00:00.000000000,0,246,0,0,0,...,0,0,0,1,1,1,0,0,0,1
869442,2015-12-28,15893,1,62,1970-01-01 00:00:00.000000726,0,251,0,0,0,...,0,0,0,0,0,1,0,0,0,0
281848,2015-05-28,15894,1,59,1970-01-01 00:00:00.000000000,0,246,1,0,0,...,0,0,0,1,1,1,0,1,1,1


In [ ]:
X_train=df.drop(columns=product_cols+['customer_id', 'date', 'join_date'])
Y_train=df[product_cols]
X_test=df_test.drop(columns=['customer_id', 'date', 'join_date'])
X_test=X_test[X_train.columns]

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
model=MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=20,
        max_depth=8,
        random_state=42,
        n_jobs=-1
        ),
    n_jobs=-1
)
model.fit(X_train, Y_train)
y_pred=model.predict(X_test)
y_pred_df=pd.DataFrame(y_pred, columns=product_cols)
print(y_pred)

[[0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]]
